# 🧠 Taller 3: Clínica Psiquiátrica IA — Técnicas de Prompt Engineering

Hoy trabajamos en una **clínica psiquiátrica** y nuestra herramienta de apoyo es un **modelo de lenguaje**. Dependiendo de cómo le pidamos las cosas, nos dará respuestas genéricas o análisis profesionales.

Cada técnica de prompting es una herramienta clínica diferente:

| Técnica | Herramienta clínica | Ejemplo |
|---|---|---|
| **Zero Shot** | Redacción clínica | Mejorar un diagnóstico mal escrito |
| **Few Shot** | Historial clínico | Casos similares como referencia |
| **Chain of Thought** | Evaluación diagnóstica | Análisis paso a paso |
| **Role Prompting** | Junta médica | Diferentes especialistas opinan |

⏱️ **Tiempo estimado:** 30 minutos

> ⚠️ **Nota:** Este taller es con fines educativos sobre prompt engineering. La IA no reemplaza profesionales de salud mental.

In [ ]:
import ollama

def consultar(prompt, opciones=None, modelo="gemma4:e2b"):
    respuesta = ollama.generate(
        model=modelo,
        prompt=prompt,
        options=opciones or {},
        think=False
    )
    print("=" * 60)
    print(f"Parametros: {opciones or 'default'}")
    print("=" * 60)
    print(respuesta["response"])
    print(f"\n[Tokens generados: {respuesta.get('eval_count', 'N/A')}]")
    return respuesta["response"]

print("Consultorio listo. Sistema IA conectado.")

In [ ]:
# Prueba de conexion
respuesta = ollama.generate(
    model="gemma4:e2b",
    prompt="Necesito que respondas unicamente el siguiente texto 'Consultorio Psicologico IA listo' y nada mas.",
    think=False
)
print(respuesta["response"])

## 1️⃣ Zero Shot: Redacción Clínica

Vamos pedirle al modelo que haga algo **sin darle ejemplos**. Solo con la instrucción.

📝 Un residente escribió este diagnóstico a las 3am después de una guardia. Vamos a pedirle a la IA que lo mejore.

In [ ]:
# ZERO SHOT: Mejorar la redaccion de un diagnostico
diagnostico_mal_escrito = """paciente juan lopez, 32 años vino pq no puede dormir
hace como 3 semanas y dice q esta re ansioso en el camello, no tiene historial
de nada psiquiatrico. se le ve nervioso y habla rapido. creo q es ansiedad
generalizada o algo asi, le mande unas pastillas para dormir y que vuelva
en 2 semanas a ver como sigue."""

prompt_zero_shot = f"""Eres un psiquiatra clinico redactando informes para el expediente
de la clinica.

El siguiente diagnostico fue escrito de forma informal por un pasante de psiquiatria.
Reescribelo como un informe clinico profesional.

Diagnostico original:
{diagnostico_mal_escrito}
"""

print("ZERO SHOT - Mejorar redaccion de diagnostico")
print()
consultar(prompt_zero_shot, opciones={"temperature": 0.3, "num_predict": 2000})

## 2️⃣ Few Shot: Historial Clínico

En muchos casos, varias industrias estandarizan formatos para facilitar la lectura y auditoria de los casos, ahora daremos **ejemplos de casos anteriores** antes de pedirle que analice un caso nuevo para que La IA aprende el patrón y nos de en el formato que usa clínica.

👤 **Paciente Pedro:** 45 años, se siente triste y sin motivación desde hace 2 meses. Vamos a darle casos similares para que formule una recomendación consistente.

In [ ]:
# FEW SHOT: Paciente Pedro con historial de casos similares
prompt_pedro = """Eres un psicologo clinico. Analiza pacientes y formula recomendaciones
siguiendo el formato de estos casos anteriores:

Caso 1 - Maria, 38 anos: Fatiga constante, perdida de interes en actividades.
Diagnostico: Episodio depresivo leve.
Recomendacion: Terapia cognitivo-conductual (12 sesiones) + rutina de ejercicio 3x/semana.

Caso 2 - Carlos, 52 anos: Irritabilidad, dificultad para concentrarse, dolores de cabeza.
Diagnostico: Sindrome de burnout.
Recomendacion: Licencia laboral temporal + tecnicas de manejo de estres + evaluacion en 4 semanas.

Caso 3 - Lucia, 29 anos: Llanto frecuente, aislamiento social, perdida de apetito tras ruptura sentimental.
Diagnostico: Trastorno adaptativo con animo depresivo.
Recomendacion: Terapia de apoyo semanal + actividades de reconexion social + seguimiento en 6 semanas.

Ahora analiza este nuevo paciente:

Paciente Pedro, 45 anos: Se siente triste y sin motivacion desde hace 2 meses.
Ha dejado de hacer ejercicio, duerme mas de lo normal y ha perdido interes en
su trabajo como profesor. No identifica un evento desencadenante claro.

Usa el mismo formato: Diagnostico + Recomendacion."""

print("FEW SHOT - Paciente Pedro (tristeza y desmotivacion)")
print()
consultar(prompt_pedro, opciones={"temperature": 0.3, "num_predict": 200})

## 3️⃣ Chain of Thought: Evaluación Diagnóstica

🔗 **Chain of Thought** = pedirle al modelo que **razone paso a paso** antes de dar su conclusión.

👵 **Paciente Rosa:** 68 años, su familia reporta que olvida cosas frecuentemente. ¿Es demencia o envejecimiento normal? La IA debe evaluar con 3 preguntas clínicas y razonar antes de concluir.

In [ ]:
# CHAIN OF THOUGHT: Evaluacion de paciente Rosa
prompt_rosa = """Eres un neuropsicologo clinico evaluando un posible caso de deterioro cognitivo.

Paciente: Rosa, 68 anos. Su familia reporta que olvida nombres de familiares cercanos,
pierde objetos cotidianos (llaves, telefono) y la semana pasada se desoriento
volviendo del supermercado, un camino que hace desde hace 20 anos.

Piensa paso a paso:

Paso 1: Formula 3 preguntas clinicas clave que le harias a Rosa y su familia
para diferenciar envejecimiento normal de deterioro cognitivo.

Paso 2: Para cada pregunta, explica que respuesta indicaria envejecimiento normal
y cual indicaria deterioro cognitivo.

Paso 3: Basandote en los sintomas descritos (olvido de nombres cercanos,
perdida de objetos, desorientacion en ruta familiar), da tu evaluacion preliminar.

Paso 4: Recomienda los siguientes pasos (pruebas, derivaciones, seguimiento).

Formato: Enumera cada paso claramente y explica detalladamente su justificacion.
Restriccion: Se cauteloso, esto es una evaluacion preliminar, no un diagnostico definitivo."""

print("CHAIN OF THOUGHT - Paciente Rosa (sospecha de deterioro cognitivo)")
print()
consultar(prompt_rosa, opciones={"temperature": 0.4, "num_predict": 2000})

## 4️⃣ Role Prompting: Junta Médica

🎭 **Role Prompting** = asignarle un **rol específico** al modelo. La misma situación vista por diferentes especialistas produce análisis muy distintos.

👩‍💻 **Paciente Ana:** 25 años, miedo intenso a hablar en público que le está afectando su carrera. Veamos qué dice cada especialista.

In [ ]:
# ROLE PROMPTING: 3 especialistas evaluan a la paciente Ana
caso_ana = """Paciente: Ana, 25 anos, ingeniera de software. Tiene miedo intenso a hablar
en publico. Evita presentaciones en el trabajo, rechaza ascensos que impliquen liderar
reuniones, y la semana pasada tuvo un ataque de panico antes de una exposicion."""

# Especialista 1: Psicologo cognitivo-conductual
prompt_psicologo = f"""Eres un psicologo especialista en terapia cognitivo-conductual (TCC)
con enfoque en fobias y ansiedad.

{caso_ana}

Da tu analisis y plan de tratamiento en maximo 4 oraciones."""

print("ESPECIALISTA 1: Psicologo Cognitivo-Conductual")
print()
consultar(prompt_psicologo, opciones={"temperature": 0.5, "num_predict": 500})

print("\n" + "~" * 60 + "\n")

# Especialista 2: Psiquiatra
prompt_psiquiatra = f"""Eres un psiquiatra con enfoque farmacologico y neurobiologico.

{caso_ana}

Da tu analisis y plan de tratamiento en maximo 4 oraciones."""

print("ESPECIALISTA 2: Psiquiatra")
print()
consultar(prompt_psiquiatra, opciones={"temperature": 0.5, "num_predict": 500})

print("\n" + "~" * 60 + "\n")

# Especialista 3: Coach de vida
prompt_coach = f"""Eres un coach de vida y oratoria con experiencia ayudando a profesionales
a superar el miedo escenico. Tu enfoque es practico y motivacional.

{caso_ana}

Da tu analisis y plan de accion en maximo 4 oraciones."""

print("ESPECIALISTA 3: Coach de Vida y Oratoria")
print()
consultar(prompt_coach, opciones={"temperature": 0.5, "num_predict": 500})

## 5️⃣ Combinando Todo: Caso Complejo

🧩 Ahora combinamos **todas las técnicas** en un solo prompt: Role + Chain of Thought + formato estructurado.

In [ ]:
# COMBINACION: Todas las tecnicas en un solo prompt
prompt_final = """Eres un psicologo clinico senior con 20 anos de experiencia y
especializacion en trastornos del animo. Trabajas en un hospital universitario.

CASO CLINICO:
Paciente Diego, 40 anos, empresario. Desde hace 3 meses presenta:
- Insomnio severo (duerme 2-3 horas por noche)
- Irritabilidad extrema con su familia y empleados
- Perdida de 8 kg sin dieta
- Episodios de llanto sin motivo aparente
- Consumo de alcohol incrementado (de social a diario)
- Verbaliza: "No le veo sentido a nada"

Casos de referencia en tu expediente:
- Paciente similar A: Sintomas parecidos tras divorcio -> Depresion mayor, respondio a TCC + ISRS.
- Paciente similar B: Sintomas parecidos + consumo de sustancias -> Trastorno depresivo con
  comorbilidad de abuso de sustancias, requirio internacion breve.

Piensa paso a paso:
Paso 1: Identifica los sintomas clave y su gravedad
Paso 2: Compara con los casos de referencia
Paso 3: Formula tu evaluacion preliminar
Paso 4: Disena un plan de tratamiento con prioridades

Formato: Secciones numeradas por cada paso.
Restriccion: Prioriza la seguridad del paciente. Esto es una evaluacion preliminar."""

print("CASO COMPLEJO: Paciente Diego (combinacion de tecnicas)")
print()
consultar(prompt_final, opciones={
    "temperature": 0.4,
    "num_predict": 2000,
    "repeat_penalty": 1.3
})

## 6️⃣ Tu Turno: Crea tu Consulta

✏️ Inventa un paciente y aplica **al menos 2 técnicas** diferentes. Algunas ideas:
- 📚 Un estudiante con estres por examenes (Zero Shot + Role Prompting)
- 🏡 Un adulto mayor con aislamiento social (Few Shot + Chain of Thought)
- 📱 Un adolescente con adiccion al celular (Chain of Thought + Role Prompting)

In [ ]:
# Tu consulta personalizada
mi_prompt = """[ESCRIBE TU PROMPT AQUI]"""

# Descomenta cuando estes listo:
# consultar(mi_prompt, opciones={"temperature": 0.5, "num_predict": 2000})

### 📋 Resumen

| Técnica | Qué hace | Cuándo usarla |
|---|---|---|
| **Zero Shot** | Pide sin ejemplos | Tareas generales, primera consulta |
| **Few Shot** | Da ejemplos primero | Seguir un formato o patrón específico |
| **Chain of Thought** | Razona paso a paso | Análisis complejo, evaluaciones |
| **Role Prompting** | Asigna un rol | Perspectivas especializadas |

---

🏥 **Consultorio cerrado por hoy. Buen trabajo, doctores.** 👨‍⚕️👩‍⚕️